# 2b · Label Audit (human labels on a sample of test atoms)

The test atoms get their labels from the sentence they come from: notebook 1 labels every sentence, and notebook 2 passes that label to the sentence's atoms. That covers both the normal and the mixed half. With thousands of atoms, labelling all of them by hand is not feasible. Instead this notebook measures how often those labels are right, the standard way to validate silver labels:

1. **Sample** `N_AUDIT_ATOMS` atoms from `data/atomic_test.csv`, stratified by half (`gen_mode`: normal / mixed) × pipeline label. Every cell gets the same number of atoms, so rare labels (contradiction) are measured as precisely as common ones.
2. **Blind sheets.** Each of the three annotators gets a sheet in its own random order, showing only the premise, the two drugs and the atom. They never see the pipeline label, the answer it came from, which half it belongs to, or each other's labels.
3. **Import.** The notebook reports:
   - **Fleiss' κ** over the three annotators, plus Cohen's κ for each pair;
   - each annotator's agreement with the pipeline labels;
   - the **gold label**: the majority label (at least 2 of 3). Only atoms where all three disagree go to adjudication;
   - the **gold-vs-pipeline agreement per half and label**, plus an overall figure per half weighted back to the real label mix. That figure is the label-noise estimate to report in the thesis.
4. **Gold subset.** `data/atomic_test_audit.csv` holds the audited atoms (notebook 2's columns, plus `gen_mode`, the sampling `weight`, `gold_label` and `n_agree`: how many annotators gave the gold label).

Nothing in the main data files is changed. Runs after notebook 2, with no Gemini or scispaCy needed.


In [ ]:
SPLIT = "test"
N_AUDIT_ATOMS = 300              # total sample; split equally over the (gen_mode x label) cells
SEED = 21
ANNOTATORS = ["A", "B", "C"]     # one sheet each; gold = majority label (2 of 3)
OVERWRITE_SHEETS = False         # True rebuilds the sheets and key - this discards filled-in labels
AUDIT_DIR = "data/audit"
# notebook 2 diagnostics that are not needed here (older atomic files may still contain them)
DROP_COLUMNS = ["repaired", "context_trimmed", "trigger_words", "other_drugs", "filter_reason"]


## Load the test atoms

In [ ]:
import json, math, os
from itertools import combinations
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import cohen_kappa_score

LABELS = ["entailment", "contradiction", "neutral"]
os.makedirs(AUDIT_DIR, exist_ok=True)
KEY_PATH = f"{AUDIT_DIR}/audit_key.csv"
ADJ_PATH = f"{AUDIT_DIR}/adjudication.csv"

atoms = pd.read_csv(f"data/atomic_{SPLIT}.csv")
atoms = atoms.drop(columns=[col for col in DROP_COLUMNS if col in atoms.columns])
holistic = pd.read_csv(f"data/holistic_{SPLIT}_clean.csv")
mode_of = (holistic.set_index("source_row")["gen_mode"] if "gen_mode" in holistic.columns
           else pd.Series("normal", index=holistic["source_row"]))
atoms["gen_mode"] = atoms["source_row"].map(mode_of).fillna("normal")
population = atoms.groupby(["gen_mode", "label"]).size().rename("atoms")
print(f"{len(atoms):,} {SPLIT} atoms")
display(population.unstack(fill_value=0))


## Export the annotation sheets

**Annotation guideline.** Read the premise, then the atom, and write one label in `label`:
- **entailment (e):** the atom must be true if the premise is true.
- **contradiction (c):** the atom cannot be true if the premise is true. It is about the same drugs and the same measure, and the two are incompatible.
- **neutral (n):** the premise neither confirms nor rules out the atom. Added details count here: a mechanism, dose or population the premise does not mention.

Hedges matter: "may increase" is not the same claim as "increases". Swapping which drug affects which is neutral, not contradiction. Use `unsure` (y) for hard cases, and `notes` for anything else. Judge against the premise only, without looking anything up.

The sheets are written once. Re-running this cell leaves existing sheets alone unless `OVERWRITE_SHEETS = True`.

In [ ]:
def export_sheets():
    cells = atoms.groupby(["gen_mode", "label"])
    per_cell = math.ceil(N_AUDIT_ATOMS / cells.ngroups)
    sample = pd.concat([g.sample(n=min(per_cell, len(g)), random_state=SEED) for _, g in cells])
    sample = sample.assign(weight=sample.set_index(["gen_mode", "label"]).index.map(
        lambda c: population[c] / min(per_cell, population[c])))
    key = sample.sample(frac=1, random_state=SEED).reset_index(drop=True)
    key.insert(0, "atom_id", [f"A{i:04d}" for i in range(1, len(key) + 1)])
    key.to_csv(KEY_PATH, index=False)
    for k, annotator in enumerate(ANNOTATORS):
        sheet = key[["atom_id", "premise", "Entity1", "Entity2", "hypothesis"]].rename(
            columns={"Entity1": "drug_1", "Entity2": "drug_2", "hypothesis": "atom"})
        sheet = sheet.sample(frac=1, random_state=SEED + 1 + k).assign(label="", unsure="", notes="")
        sheet.insert(0, "item", range(1, len(sheet) + 1))
        sheet.to_csv(f"{AUDIT_DIR}/audit_sheet_{annotator}.csv", index=False)
    print(f"{len(key)} atoms ({per_cell} per gen_mode x label cell, fewer where a cell is smaller) -> "
          + ", ".join(f"{AUDIT_DIR}/audit_sheet_{a}.csv" for a in ANNOTATORS)
          + f"; keep {KEY_PATH} away from the annotators")
    display(key.groupby(["gen_mode", "label"]).size().unstack(fill_value=0))


if os.path.exists(KEY_PATH) and not OVERWRITE_SHEETS:
    print(f"{KEY_PATH} already exists: the sheets were exported before and may be filled in, so they are left alone. "
          f"Set OVERWRITE_SHEETS = True to rebuild them (this discards their labels).")
else:
    export_sheets()


## Import the annotations

1. **Read** the filled sheets. Labels may be written as `e`, `c` or `n`, or in full.
2. **Agreement** on the atoms all three annotators labelled: Fleiss' κ, Cohen's κ per pair, and how often 3/3 or 2/3 agree. Also each annotator against the pipeline labels.
3. **Gold:** the majority label (2 of 3). Atoms where all three disagree go to `adjudication.csv`: fill `final_label` there and run this cell again. Until every case is adjudicated the report is provisional, because the hardest atoms are still missing from it.
4. **Report:**
   - gold vs pipeline label per half and label;
   - the confusion between them;
   - overall agreement per half, weighted back to the real number of atoms in each cell.
5. **Write** `data/atomic_test_audit.csv`.


In [ ]:
ALIASES = {"e": "entailment", "ent": "entailment", "entail": "entailment", "entailment": "entailment",
           "c": "contradiction", "con": "contradiction", "contra": "contradiction", "contradiction": "contradiction",
           "n": "neutral", "neu": "neutral", "neutral": "neutral"}


def read_sheet(annotator):
    path = f"{AUDIT_DIR}/audit_sheet_{annotator}.csv"
    if not os.path.exists(path):
        return pd.Series(dtype=object)
    sheet = pd.read_csv(path, dtype=str).fillna("")
    raw = sheet.set_index("atom_id")["label"].str.strip().str.lower()
    labels = raw.map(ALIASES)
    bad = sorted(set(raw[(raw != "") & labels.isna()]))
    if bad:
        print(f"[{annotator}] unreadable labels ignored: {bad}")
    return labels


def kappa(a, b):
    return cohen_kappa_score(a, b) if len(a) and (pd.Series(a).nunique() > 1 or pd.Series(b).nunique() > 1) else float("nan")


def fleiss_kappa(votes):
    """Fleiss' kappa; rows = atoms labelled by every annotator, columns = annotators."""
    r = votes.shape[1]
    counts = np.stack([(votes == lab).sum(axis=1).to_numpy() for lab in LABELS], axis=1)
    if len(counts) == 0 or r < 2:
        return float("nan")
    p_atom = ((counts ** 2).sum(axis=1) - r) / (r * (r - 1))
    p_label = counts.sum(axis=0) / counts.sum()
    p_chance = (p_label ** 2).sum()
    return float("nan") if p_chance == 1 else (p_atom.mean() - p_chance) / (1 - p_chance)


def import_annotations():
    key = pd.read_csv(KEY_PATH).set_index("atom_id")
    key = key.drop(columns=[col for col in DROP_COLUMNS if col in key.columns])
    votes = pd.DataFrame({a: read_sheet(a) for a in ANNOTATORS}).reindex(key.index)
    merged = key.join(votes)

    for a in ANNOTATORS:
        done = merged[a].notna()
        print(f"annotator {a}: {done.sum()}/{len(merged)} atoms labelled")
        for mode, part in merged[done].groupby("gen_mode"):
            print(f"   {mode:7s} agreement with the pipeline label {(part[a] == part['label']).mean():.1%} "
                  f"(kappa {kappa(part[a], part['label']):.3f}, n={len(part)})")

    complete = votes.notna().all(axis=1)
    full = votes[complete]
    print(f"\n{len(full)}/{len(votes)} atoms labelled by all {len(ANNOTATORS)} annotators")
    if len(full):
        if len(ANNOTATORS) >= 3:
            print(f"Fleiss' kappa: {fleiss_kappa(full):.3f}")
        for a, b in combinations(ANNOTATORS, 2):
            print(f"Cohen's kappa {a} vs {b}: {kappa(full[a], full[b]):.3f} (raw agreement {(full[a] == full[b]).mean():.1%})")
        top = full.apply(lambda row: row.value_counts().iloc[0], axis=1)
        print("annotators agreeing on the most common label:",
              ", ".join(f"{k}/{len(ANNOTATORS)}: {v}" for k, v in top.value_counts().sort_index(ascending=False).items()))
        merged["n_agree"] = top
        majority = full.apply(lambda row: row.value_counts().index[0], axis=1).where(top > len(ANNOTATORS) / 2)
        merged["gold_label"] = majority
    else:
        merged["n_agree"], merged["gold_label"] = np.nan, np.nan

    no_majority = complete & merged["gold_label"].isna()
    previous = (pd.read_csv(ADJ_PATH, dtype=str).fillna("").set_index("atom_id")["final_label"]
                if os.path.exists(ADJ_PATH) else pd.Series(dtype=str))
    adjudicated = previous.str.strip().str.lower().map(ALIASES).dropna()
    adjudicated = adjudicated[adjudicated.index.isin(merged.index[no_majority])]
    merged.loc[adjudicated.index, "gold_label"] = adjudicated
    open_cases = 0
    if no_majority.any():
        sheet = merged.loc[no_majority, ["premise", "hypothesis"] + ANNOTATORS].copy()
        sheet["final_label"] = previous.reindex(sheet.index).fillna("")
        sheet.reset_index().to_csv(ADJ_PATH, index=False)
        open_cases = int((sheet["final_label"].str.strip() == "").sum())
        print(f"\n{no_majority.sum()} atoms without a majority in {ADJ_PATH}"
              + (f": {open_cases} still need a final_label (fill it, then re-run)" if open_cases else ", all adjudicated"))

    gold = merged[merged["gold_label"].notna()]
    if gold.empty:
        print("\nno gold labels yet")
        return
    print(f"\n{len(gold)}/{len(merged)} atoms have a gold label")
    if open_cases or not complete.all():
        print("PROVISIONAL: atoms that are unlabelled or not yet adjudicated are left out, and they tend to be the "
              "hard ones, so the agreement below is too high.")
    gold = gold.assign(agree=gold["gold_label"] == gold["label"])
    print("\ngold agrees with the pipeline label (share, n) per half and pipeline label:")
    display(gold.groupby(["gen_mode", "label"])["agree"].agg(["mean", "count"]).round(3))
    for mode, part in gold.groupby("gen_mode"):
        weighted = np.average(part["agree"], weights=part["weight"])
        print(f"{mode}: agreement weighted to the real label mix {weighted:.1%} "
              f"(kappa {kappa(part['gold_label'], part['label']):.3f})")
        display(pd.crosstab(part["label"].rename("pipeline"), part["gold_label"].rename("gold")))
    out = gold.drop(columns=ANNOTATORS + ["agree"]).reset_index()
    out.to_csv(f"data/atomic_{SPLIT}_audit.csv", index=False)
    print(f"wrote data/atomic_{SPLIT}_audit.csv ({len(out)} atoms with gold_label)")


if os.path.exists(KEY_PATH):
    import_annotations()
else:
    print(f"{KEY_PATH} not found: export the sheets first")
